# Kahupuke Prototype BERT


This notebook is a quick and dirty BERT pipeline for Hawaiian (ʻŌlelo Hawaiʻi) using `bert-base-multilingual-uncased`. For documentation purposes, this is used to bootstrap the actual BERT pipeline with an official dataset. This is for educative purposes

TODO: This BERT was trained on a Bilingual Lexicon Dataset. We need to swap to an actual complete sentence Dataset derived from ʻŌlelo Hawaiʻi. Though, this serves as a working pipeline with interactive 


In [28]:
import sys
import os
import json
import pandas as pd
import math, numpy as np, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel


UTIL_PATH = os.path.abspath("./util.py")
if UTIL_PATH not in sys.path:
    sys.path.append(UTIL_PATH)

from util import InternalState  # Replace with the actual class or function


In [29]:
api = InternalState()
data = api.hawaiian_to_english

In [30]:
data = json.loads(pretty_json)
records = []
for k, rows in data.items():         # k = unpredictable key
    for row in rows:                 # row is the inner list
        records.append([k] + row)    # prepend the key so we don't lose it


In [31]:
df = pd.DataFrame(records)
df.columns = ["source"] + [f"col{i}" for i in range(df.shape[1] - 1)]
df = df.iloc[:, 1:]                    # or: df = df.drop(columns=0)
df.columns = ["haw", "eng"]

In [32]:
df.sample(5)

,haw,eng
28680,luna mālama waiwai,trustee
16261,ʻōʻaki,"geometrid moth, moth"
7479,waiwai pio,"booty, loot, plunder, spoil"
5900,palena holo kani,"barrier, sound"
1167,Akua Kahikolu,Holy Trinity


In [33]:
torch.set_grad_enabled(False)

MODEL_NAME = "bert-base-multilingual-uncased"
BATCH      = 128

tok  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME, device_map="auto").eval()

def embed_batch(text_batch):
    enc = tok(text_batch, padding=True, truncation=True, return_tensors="pt")
    enc = {k: v.to(bert.device) for k, v in enc.items()}
    out = bert(**enc).last_hidden_state[:, 0, :]
    return out.cpu().numpy()

total_steps = math.ceil(len(df) / BATCH)

In [34]:
haw_vecs = np.empty((len(df), bert.config.hidden_size), dtype=np.float32)
for step, i in enumerate(
        tqdm(range(0, len(df), BATCH),
             total=total_steps,
             desc="Embedding Hawaiian",
             unit="batch")):
    batch = df["haw"].iloc[i : i + BATCH].tolist()
    haw_vecs[i : i + len(batch)] = embed_batch(batch)


Embedding Hawaiian:   0%|          | 0/234 [00:00<?, ?batch/s]

In [35]:
eng_vecs = np.empty_like(haw_vecs)
for step, i in enumerate(
        tqdm(range(0, len(df), BATCH),
             total=total_steps,
             desc="Embedding English",
             unit="batch")):
    batch = df["eng"].iloc[i : i + BATCH].tolist()
    eng_vecs[i : i + len(batch)] = embed_batch(batch)

Embedding English:   0%|          | 0/234 [00:00<?, ?batch/s]

In [36]:
print("Hawaiian vectors:", haw_vecs.shape
      , "English vectors:", eng_vecs.shape)

Hawaiian vectors: (29833, 768) English vectors: (29833, 768)


In [37]:
import itertools, pandas as pd, pathlib, random, os

sample_lines = df["haw"].dropna().tolist()[:2000]
pathlib.Path("quick/train.txt").write_text("\n".join(sample_lines))
pathlib.Path("quick/valid.txt").write_text("\n".join(sample_lines[:200]))  # 10%

from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
torch.set_grad_enabled(True)

MODEL = "distilbert-base-multilingual-cased"
tok   = AutoTokenizer.from_pretrained(MODEL)

ds = load_dataset("text",
    data_files={"train": "quick/train.txt", "validation": "quick/valid.txt"})

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=128)
token_ds = ds.map(tokenize, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)

# Continue-pre-train for 300 steps (≈ 5 min GPU)
from transformers import AutoModelForMaskedLM, Trainer, TrainingArguments
model = AutoModelForMaskedLM.from_pretrained(MODEL)

args = TrainingArguments(
    "quick-haw-mlm",
    max_steps=300,
    per_device_train_batch_size=32,
    learning_rate=5e-5,
    save_total_limit=1,
    eval_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(model, args,
                  train_dataset=token_ds["train"],
                  data_collator=collator)
trainer.train()

model.save_pretrained("quick-haw-mlm")
tok.save_pretrained("quick-haw-mlm")
print("done — mini Hawaiian MLM ")


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Step,Training Loss


done — mini Hawaiian MLM 


#  Hawaiian MLM Interactive Prompt

This is an interactive prompt for using 
the Hawaiian Masked Language Model 
(MLM). Follow these instructions to get 
started:

1. **Run the Cell**: Execute the cell 
containing the code by clicking on it 
and pressing `Shift + Enter`.

2. **Type Your Sentence**: When 
prompted, type a sentence that contains 
the `[MASK]` token where you want the 
model to predict the most likely 
word(s). For example:
   ```
   » Aloha [MASK].
   ```

3. **View Results**: The code will use 
the pre-trained Hawaiian MLM model to 
fill in the mask and print the top 5 
predictions along with their 
probabilities.

4. **Exit the Loop**: To exit the 
interactive loop, type `/exit` followed 
by pressing `Enter`. This will terminate 
the program gracefully.


In [38]:
from transformers import pipeline

fm = pipeline(
    "fill-mask",
    model="quick-haw-mlm",
    tokenizer="quick-haw-mlm",
    top_k=5,
    device_map="auto",
)

print("Hawaiian MLM • type a sentence containing [MASK] (or /exit to quit)")
while True:
    try:
        sent = input("» ")
        if sent.strip().lower() == "/exit":
            print("Exiting...")
            break
        for out in fm(sent):
            print(f" {out['token_str']:15} (p={out['score']:.3f})")
    except KeyboardInterrupt:print("\nExiting...")
    except EOFError:
        print("\nExiting...")
        break
    except Exception as e:
        print(f"Error: {e}")

Device set to use cuda:0


Hawaiian MLM • type a sentence containing [MASK] (or /exit to quit)
 ##na            (p=0.175)
 ##ni            (p=0.161)
 ##nia           (p=0.110)
 ##no            (p=0.066)
 ##ʻi            (p=0.028)
Exiting...
